# 02 — Quality control, spatially

**Day 1, 11:00–12:00**

### Where we are going

Everything you know about scRNA-seq QC applies here, and none of it is sufficient.
The extra questions are:

- **Is the chemistry clean?** Answered by the negative controls.
- **Is the segmentation sane?** Answered by counts-vs-area, and by looking.
- **Are the problems spread evenly, or are they in one corner of the slide?**
  Answered only by plotting QC **in space** — and this is the move that people
  coming from scRNA-seq forget.

By the end you can:

1. estimate the background rate from negative control probes and codewords
2. build per-cell QC metrics that are meaningful for imaging-based data
3. spot segmentation failures from the counts–area relationship
4. map every QC metric onto the tissue and recognise technical artefacts
5. choose filters and defend them — including the ones you decided *not* to apply

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=90, frameon=False)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

NAVY, GOLD, CORAL, ICE = "#001158", "#FBAE40", "#F26B43", "#BCD2FF"

In [ ]:
adata = sc.read_h5ad(DATA / "ovarian_subset.h5ad")
adata.layers["counts"] = adata.X.copy()      # keep raw counts safe, always
adata

In [ ]:
def flag_controls(adata, verbose=True):
    """Which features are controls rather than targeted genes?

    `feature_types` from the 10x file is authoritative: anything that is not
    "Gene Expression" is a control of some kind. Name matching is a fallback,
    because control naming has changed between Xenium chemistries — and a
    detector that silently finds nothing is worse than one that finds too much.
    """
    names = adata.var_names.str.lower()
    by_name = (
        names.str.startswith("negcontrol") | names.str.startswith("neg_control")
        | names.str.startswith("unassignedcodeword")
        | names.str.startswith("unassigned_codeword")
        | names.str.startswith("deprecatedcodeword")
        | names.str.startswith("genomiccontrol")
        | names.str.startswith("genomic_control")
        | names.str.startswith("antisense") | names.str.startswith("blank")
        | names.str.contains("codeword")
    )
    by_name = pd.Series(np.asarray(by_name), index=adata.var_names)

    if "feature_types" in adata.var.columns:
        ft = adata.var["feature_types"].astype(str).str.strip().str.lower()
        control = ~ft.isin(["gene expression", "gene_expression"]) | by_name
    else:
        control = by_name

    if verbose:
        n = int(control.sum())
        print(f"{n} control features, {int((~control).sum())} targeted genes")
        if n:
            print("examples:", list(adata.var_names[control.to_numpy()][:5]))
        else:
            print("\n*** No controls found — the rest of section 1 cannot run. ***")
            print("var columns:", list(adata.var.columns))
            if "feature_types" in adata.var.columns:
                print(adata.var["feature_types"].value_counts())
            print("first 10 feature names:", list(adata.var_names[:10]))
            print("\nMost likely cause: the matrix was built with scanpy's")
            print("read_10x_h5(gex_only=True) default, which keeps only")
            print("'Gene Expression' features and discards every control.")
            print("The controls are in the original .h5 but not in this file, so")
            print("no code here can recover them — the data must be regenerated.")
            print("Tell the organiser; section 1 is the only part affected.")
    return control


# Recompute rather than trusting the stored column: older versions of the data
# preparation script flagged controls by name only, and on some panels that
# finds nothing.
adata.var["control"] = flag_controls(adata)

## 1. Background: what do the negative controls say?

Two independent estimates of how much of your signal is noise.

If a negative control **probe** fires, chemistry is binding where it should not.
If a negative control **codeword** fires, the decoder is hallucinating.
Splitting them tells you *which* part of the pipeline is misbehaving.

The catch, and the reason this section is longer than you might expect: the matrix
contains several other non-gene feature classes that are **not** negative controls.
Treating them as such is the commonest way to convince yourself a clean run is
broken.

In [ ]:
import scipy.sparse as sp

X = adata.layers["counts"]
tot = np.asarray(X.sum(axis=0)).ravel()      # total counts per feature
is_ctrl = adata.var["control"].to_numpy()

if not is_ctrl.any():
    raise RuntimeError(
        "No control features in this object — see the diagnostic printed above. "
        "The rest of section 1 measures background from the controls, so it "
        "cannot run. Skip to section 2; everything from there on works fine."
    )

# Not every non-gene feature measures background. Only these two are designed
# as negative controls:
#   Negative Control Probe    - a probe against a sequence not in the tissue
#   Negative Control Codeword - a codeword no probe uses
# The others mean different things and are reported separately:
#   Unassigned Codeword  - a valid codeword with no gene assigned to it
#   Genomic Control      - probe binding genomic DNA rather than mRNA
#   Deprecated Codeword  - codewords the pipeline does not use; NOT a control
BACKGROUND_CLASSES = ["Negative Control Probe", "Negative Control Codeword"]

ftypes = (adata.var["feature_types"].astype(str)
          if "feature_types" in adata.var.columns
          else pd.Series("unknown", index=adata.var_names))
is_background = ftypes.isin(BACKGROUND_CLASSES).to_numpy() & is_ctrl
is_other = is_ctrl & ~is_background

if not is_background.any():
    print("No designated negative-control features found; falling back to all")
    print("non-gene features, which will overestimate background.\n")
    is_background = is_ctrl
    is_other = np.zeros_like(is_ctrl)

gene_tot = tot[~is_ctrl]
bg_tot = tot[is_background]

print(f"targeted genes    : {len(gene_tot):>6,} features")
print(f"negative controls : {len(bg_tot):>6,} features   <- background comes from these")
print(f"other non-gene    : {int(is_other.sum()):>6,} features   (separate table below)")

print(f"\nmean counts per targeted gene    : {gene_tot.mean():9.2f}")
print(f"mean counts per negative control : {bg_tot.mean():9.2f}")
print(f"\nbackground rate, mean ratio      : {bg_tot.mean() / gene_tot.mean():9.4f}")
print(f"background rate, median ratio    : "
      f"{np.median(bg_tot) / max(np.median(gene_tot), 1):9.4f}")

### Not every non-gene feature is a control

This distinction matters and is easy to miss. The matrix holds up to five kinds of
non-gene feature, and only two of them measure background:

| Class | What it is | Use for background? |
|---|---|---|
| **Negative Control Probe** | probe against a sequence absent from the tissue | **yes** — measures chemistry |
| **Negative Control Codeword** | a codeword no probe uses | **yes** — measures decoding |
| Unassigned Codeword | valid codeword, no gene assigned | a rough decoding check |
| Genomic Control | probe binding genomic DNA, not mRNA | no — measures sample prep |
| Deprecated Codeword | codewords the pipeline does not use | **no — not a control at all** |

Deprecated codewords are the trap. 10x describes them as codewords not used by the
onboard analysis pipeline: retired from the active panel but still present in the
codebook. They can accumulate enormous counts, and including them in a background
estimate inflates it by orders of magnitude — turning a clean run into an apparent
disaster.

All five are still removed from the count matrix; none is a panel gene. The
distinction is purely about what you use to *estimate background*.

In [ ]:
# Per-class breakdown. Each class fails for a different reason, so whichever
# class is elevated tells you where to look.
rows = []
for cls in sorted(ftypes[is_ctrl].unique()):
    m = (ftypes == cls).to_numpy() & is_ctrl
    rows.append({
        "class": cls,
        "n features": int(m.sum()),
        "total counts": int(tot[m].sum()),
        "median/feature": round(float(np.median(tot[m])), 1),
        "max/feature": int(tot[m].max()),
        "% of all counts": round(100 * tot[m].sum() / tot.sum(), 3),
        "background?": "yes" if cls in BACKGROUND_CLASSES else "no",
    })
display(pd.DataFrame(rows).set_index("class"))

dep = (ftypes == "Deprecated Codeword").to_numpy() & is_ctrl
if dep.any() and tot[dep].sum() > 3 * max(tot[is_background].sum(), 1):
    print("Deprecated codewords carry far more counts than the true negative")
    print("controls here. That is common and is NOT a quality problem — they are")
    print("not controls. Judge this run on the two negative-control rows only.")

**How to read that background rate.** It is roughly "what fraction of an average
gene's signal could be noise". Calibrate against what Xenium actually achieves, not
against scRNA-seq intuition:

| Background rate | Verdict |
|---|---|
| ~0.0001 (1e-4) | a good Prime run — negative controls barely fire at all |
| ~0.001 | still fine |
| ~0.01 | worth investigating; weak genes are becoming unreliable |
| >0.05 | something is wrong with the chemistry or the section |

A typical clean run puts a few dozen counts in total across ~650 negative-control
features, against millions of counts on genes. If your rate is around 1e-4, background
is **not** your limiting problem — statistical power is. That changes what the
threshold below is for.

Compare the mean and median versions. If they disagree a lot, a handful of extreme
features is doing the damage rather than a raised floor.

> **Panel design still bites.** Genes at the bottom of the dynamic range are the ones
> people are most excited about, and the ones you can say least about — but on a clean
> run the reason is *too few counts to estimate anything*, not contamination.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
bins = np.logspace(0, np.log10(max(tot.max(), 10)), 70)
ax.hist(np.clip(gene_tot, 1, None), bins=bins, color=NAVY, alpha=0.85,
        label=f"targeted genes (n={len(gene_tot):,})")
if is_other.any():
    ax.hist(np.clip(tot[is_other], 1, None), bins=bins, color="0.6", alpha=0.6,
            label=f"other non-gene (n={int(is_other.sum()):,})")
ax.hist(np.clip(bg_tot, 1, None), bins=bins, color=CORAL, alpha=0.95,
        label=f"negative controls (n={len(bg_tot):,})")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("total counts in crop"); ax.set_ylabel("features")
ax.legend(fontsize=8)
ax.set_title("Only the orange distribution measures background")
sns.despine(); plt.show()

floor = np.percentile(bg_tot, 99)
n_below = int((gene_tot < floor).sum())
print(f"99% of negative controls sit below {floor:.0f} counts.")
print(f"{n_below} targeted genes ({100 * n_below / len(gene_tot):.1f}%) fall below "
      f"that — treat them as not measurable in this crop.")

### Reading this figure

Three separate things are visible, and only one of them is a problem.

**The wall at exactly 1 count.** A Prime 5K codebook contains far more codewords than
it has genes, so thousands of unassigned codewords sit in the matrix, each firing
essentially never. Thousands of features with one count each is the *expected*
picture and is good news: the decoder almost never invents a codeword. Do not read the
height of that bar as "lots of background" — it is lots of *features*, each carrying
almost nothing.

**The scattered controls between roughly 2 and 30 counts.** This is the real
background floor. Compare it with where your genes of interest sit.

**The handful of controls out at 10⁴–10⁵ counts.** *This* is the part worth
investigating. A few individual control features carrying more counts than most real
genes is not ordinary background — it is usually one probe cross-hybridising, or one
codeword a single bit away from a highly expressed gene. Those few features also drag
the mean upwards, which is why the summary statistic above can look alarming when the
bulk of the distribution is fine.

The next cell names them.

In [ ]:
# Which non-gene features carry the counts? Almost always a short list.
ctrl_counts = pd.Series(tot[is_ctrl], index=adata.var_names[is_ctrl]).sort_values(ascending=False)
top = ctrl_counts.head(10).to_frame("counts")
top["class"] = ftypes.reindex(top.index)
top["x median gene"] = (top["counts"] / max(np.median(gene_tot), 1)).round(1)
top["is a real control"] = top["class"].isin(BACKGROUND_CLASSES).map({True: "yes", False: "no"})
display(top)

n_ones = int((ctrl_counts == 1).sum())
print(f"{n_ones:,} non-gene features have exactly 1 count (of {len(ctrl_counts):,})")

if (~top["class"].isin(BACKGROUND_CLASSES)).all():
    print("\nNone of the top ten is a negative control — so none of them tells you")
    print("anything about background. Look at the negative-control row of the table")
    print("above instead.")

In [ ]:
# Mean vs median for the two distributions that matter.
summary = pd.DataFrame({
    "targeted genes": [len(gene_tot), gene_tot.mean(), np.median(gene_tot),
                       np.percentile(gene_tot, 95), gene_tot.max()],
    "negative controls": [len(bg_tot), bg_tot.mean(), np.median(bg_tot),
                          np.percentile(bg_tot, 95), bg_tot.max()],
}, index=["n features", "mean", "median", "95th pct", "max"]).round(2)
display(summary)

# Where do specific genes sit relative to the background floor?
floor = np.percentile(bg_tot, 99)
for g in ["EPCAM", "TOP2A", "TRAC", "CXCR4"]:
    if g in adata.var_names:
        v = tot[adata.var_names.get_loc(g)]
        verdict = "below the control floor — do not trust it" if v < floor else \
                  ("close to the floor — cluster means only" if v < 3 * floor
                   else "well above background")
        print(f"  {g:<8} {v:>9,.0f} counts   {verdict}")

### What to do about the outlier controls

They are not a reason to discard the dataset. Two practical responses:

1. **Note them, and check your genes of interest are not among the affected.** One
   cross-hybridising probe says nothing about the rest of the panel.
2. **If an outlier is an *unassigned codeword***, it may be one bit away from a highly
   expressed gene in the codebook. That is decoding bleed, and it means that gene's
   own counts are slightly inflated too.

What would genuinely worry you is the opposite shape: the whole orange distribution
shifted right, overlapping the middle of the blue one. That is a run with systematic
background, and no downstream filtering repairs it.

### Exercise 2.1b
Take the single largest control feature. How many targeted genes in this panel have
*fewer* counts than it? Those genes are, in this section, less reliably measured than
a feature that is supposed to measure nothing.

In [ ]:
# your code here

### Two different thresholds, and you need both

"Which genes can I use?" is really two questions.

**1. Is this gene above background?** The negative controls answer this, because they
are features that measure nothing. For a candidate threshold T: the fraction of
controls reaching T estimates the chance a pure-noise feature reaches T; multiply by
the number of genes for expected false positives; divide by genes actually above T.
That is an empirical **false discovery rate**, with the controls playing the role of a
permutation null.

**2. Is this gene usable?** A different question entirely. On a clean Prime run the
FDR threshold lands at two or three counts — statistically defensible and practically
useless, because a gene with three counts across twenty thousand cells cannot support
a mean, a fold change or a spatial statistic.

So compute both. Question 1 protects you from noise. Question 2 protects you from
underpowered claims, and on a clean run **it is the binding constraint**.

In [ ]:
def detection_fdr(gene_counts, control_counts, thresholds=None):
    """Empirical FDR for calling a gene 'detected', using controls as the null."""
    if thresholds is None:
        hi = max(int(np.percentile(control_counts, 100)) * 3, 20)
        thresholds = np.unique(np.round(np.logspace(0, np.log10(hi), 40)).astype(int))

    rows = []
    for T in thresholds:
        p = float((control_counts >= T).mean())     # P(noise feature reaches T)
        expected_false = p * len(gene_counts)
        kept = int((gene_counts >= T).sum())
        rows.append({
            "threshold": int(T),
            "controls >= T": int((control_counts >= T).sum()),
            "genes kept": kept,
            "expected false": round(expected_false, 1),
            "FDR": round(min(expected_false / max(kept, 1), 1.0), 4),
        })
    return pd.DataFrame(rows)


fdr = detection_fdr(gene_tot, bg_tot)
display(fdr.head(15))

resolution = 1 / len(bg_tot)
print(f"\nYou have {len(bg_tot)} negative controls, so the smallest non-zero noise")
print(f"probability you can measure is 1/{len(bg_tot)} = {resolution:.4f}, i.e. about")
print(f"{resolution * len(gene_tot):.0f} expected false positives. You cannot ask for a")
print("finer FDR than that — the controls simply do not have the resolution.")

In [ ]:
TARGET_FDR = 0.05        # <-- CHANGE THIS (try 0.10, 0.05, 0.01)

ok = fdr[fdr["FDR"] <= TARGET_FDR]
if len(ok) == 0:
    raise RuntimeError(
        f"No threshold reaches FDR <= {TARGET_FDR} with {len(bg_tot)} controls. "
        "Relax the target, or accept that this panel cannot support it."
    )
MIN_GENE_COUNTS = int(ok["threshold"].iloc[0])
kept = int(ok["genes kept"].iloc[0])

print(f"threshold for FDR <= {TARGET_FDR}: {MIN_GENE_COUNTS} counts in the crop")
print(f"keeps {kept:,} of {len(gene_tot):,} genes "
      f"({100 * kept / len(gene_tot):.1f}%)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(fdr["threshold"], fdr["FDR"], color=NAVY, lw=2, label="estimated FDR")
ax.axhline(TARGET_FDR, color=GOLD, lw=2, ls="--", label=f"target {TARGET_FDR}")
ax.axvline(MIN_GENE_COUNTS, color=CORAL, lw=2, label=f"threshold = {MIN_GENE_COUNTS}")
ax.set_xscale("log"); ax.set_xlabel("threshold (total counts in crop)")
ax.set_ylabel("estimated FDR"); ax.legend(fontsize=8)

ax2 = ax.twinx()
ax2.plot(fdr["threshold"], fdr["genes kept"], color="0.6", lw=1.4, ls=":")
ax2.set_ylabel("genes kept", color="0.5")
sns.despine(right=False); plt.show()

### Flag the genes — do not delete them

Two reasons to add a column rather than subset the object.

**Deleting changes the arithmetic.** `total_counts` per cell is computed over whatever
genes are present, so removing genes shifts every normalisation and every QC number
you already looked at.

**Low total counts does not mean noise.** A marker of a rare population — the ciliated
cells in this section are a few hundred cells — has low counts *in total* while being
completely real and highly expressed in the cells that have it. A total-count
threshold is biased against exactly the rare biology you might care about most.

So flag, then decide per analysis: use all genes for clustering, where a weak gene
contributes little either way, and restrict to the reliable set when you report
differential expression or interpret a single gene.

In [ ]:
adata.var["above_background"] = False
adata.var.loc[~adata.var["control"], "above_background"] = (
    tot[~adata.var["control"].to_numpy()] >= MIN_GENE_COUNTS
)
adata.uns["detection_threshold"] = {
    "min_counts": MIN_GENE_COUNTS,
    "target_fdr": TARGET_FDR,
    "n_negative_controls": int(len(bg_tot)),
}
print(adata.var.loc[~adata.var["control"], "above_background"].value_counts())

# Rescue check: is a gene below the threshold nevertheless concentrated in a few
# cells? That is the signature of a rare cell type, not of noise.
below = (~adata.var["above_background"]) & (~adata.var["control"])
if below.any():
    Xb = adata.layers["counts"][:, below.to_numpy()]
    n_cells_pos = np.asarray((Xb > 0).sum(axis=0)).ravel()
    mx = Xb.max(axis=0)
    max_in_cell = np.asarray(mx.todense() if sp.issparse(mx) else mx).ravel()
    rescue = pd.DataFrame({
        "total": tot[below.to_numpy()],
        "cells detected": n_cells_pos,
        "max in one cell": max_in_cell,
    }, index=adata.var_names[below.to_numpy()])
    # concentrated = few cells, but several copies in those cells
    rescue["concentration"] = rescue["max in one cell"] / rescue["total"].clip(lower=1)
    candidates = rescue.query("total >= 5").sort_values("concentration", ascending=False)
    print("\nBelow threshold, but concentrated in few cells — check these by hand:")
    display(candidates.head(10))

### The usability threshold

Total counts are the wrong currency for this, because a gene concentrated in a rare
population has few total counts and is perfectly usable *within* that population. The
right question is **in how many cells is it detected at all** — that is what sets
whether you can compute anything about it.

Rules of thumb, to adapt rather than obey:

- **detected in ≥ 3% of cells** — safe for per-cluster means and differential expression
- **detected in ≥ 30 cells** — the floor for saying anything at all, and only about
  the population those cells belong to
- **fewer than that** — you can report the transcripts as observed, but not a statistic

In [ ]:
n_cells_detected = np.asarray((adata.layers["counts"] > 0).sum(axis=0)).ravel()
frac_cells = n_cells_detected / adata.n_obs
adata.var["n_cells_detected"] = n_cells_detected
adata.var["frac_cells_detected"] = frac_cells

MIN_FRAC_CELLS = 0.03        # <-- CHANGE THIS (try 0.01, 0.03, 0.10)
MIN_CELLS = 30

genes_only = ~adata.var["control"].to_numpy()
usable = genes_only & (frac_cells >= MIN_FRAC_CELLS) & (n_cells_detected >= MIN_CELLS)
adata.var["usable"] = usable

n_genes_total = int(genes_only.sum())
print(f"above background (FDR)          : {int(adata.var['above_background'].sum()):>5,} / {n_genes_total:,}")
print(f"usable (>= {100 * MIN_FRAC_CELLS:.0f}% of cells)         : {int(usable.sum()):>5,} / {n_genes_total:,}")
print(f"both                            : "
      f"{int((usable & adata.var['above_background'].to_numpy()).sum()):>5,}")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(np.clip(tot[genes_only], 1, None), np.clip(frac_cells[genes_only], 1e-5, None),
           s=3, c=NAVY, alpha=0.35, linewidths=0, rasterized=True)
ax.axvline(MIN_GENE_COUNTS, color=CORAL, lw=2,
           label=f"above background: {MIN_GENE_COUNTS} counts")
ax.axhline(MIN_FRAC_CELLS, color=GOLD, lw=2,
           label=f"usable: {100 * MIN_FRAC_CELLS:.0f}% of cells")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("total counts in crop"); ax.set_ylabel("fraction of cells detected in")
ax.legend(fontsize=8, loc="lower right")
ax.set_title("the two thresholds do different work")
sns.despine(); plt.show()

### Read the scatter

The two lines rarely coincide, and where they diverge tells you something.

- **Right of the orange line, below the gold** — above background, but detected in too
  few cells to analyse. Either a rare-cell-type marker (interesting, and analysable
  *within* that population) or a gene expressed at a trickle everywhere.
- **Above the gold line, left of the orange** — detected broadly but at very low counts.
  On a clean run this is unusual; if you see many, look again at the background.
- The cloud running diagonally is the normal relationship: more counts, more cells.

**Which do you report?** Say both in the methods. "Genes were required to exceed the
negative-control null at FDR < 0.05 (≥ N counts) and to be detected in ≥ 3% of cells"
is one sentence, and it tells a reader exactly what your gene space was.

### Exercise 2.1d
Find the genes that pass the background test but fail the usability test. Are any of
them markers you would have wanted? Then check whether they are concentrated in one
region of the tissue — a gene detected in 1% of cells that are all in one place is a
very different object from one detected in 1% of cells scattered at random.

In [ ]:
# your code here

### The cross-check worth remembering

There is a second, independent way to ask whether a weak gene is real, and it is only
available because this is spatial data: **a real gene is spatially structured; noise
is not.**

In notebook 04 you compute Moran's I for every gene. A gene sitting just below your
count threshold but with clearly non-random spatial autocorrelation is almost
certainly real — noise does not form patches. A gene above the threshold with Moran's
I near zero deserves more suspicion than its count suggests.

That check has no equivalent in dissociated data, and it is a better arbiter than any
count cutoff.

### Exercise 2.1c
Set `TARGET_FDR` to 0.10 and to 0.01. How many genes does each keep? Then find one
gene that is included at 0.10 but excluded at 0.01, and decide from its spatial
pattern which call you believe.

### Exercise 2.1
Pick three genes you would actually want to use in an ovarian tumour study
(e.g. `MKI67`, `CD8A`, `PDCD1`). Where does each sit relative to the control
distribution in this crop? Would you trust a per-cell measurement of it, a
per-cluster mean, or neither?

In [ ]:
# your code here

## 2. Per-cell metrics

Now split the controls out of the matrix, then compute QC. **Order matters** — if
you run `calculate_qc_metrics` with the controls still in, your `total_counts` is
contaminated.

In [ ]:
# Per-cell background must come from the TRUE negative controls only. Summing
# every non-gene feature instead lumps in deprecated codewords, which are retired
# probes still detecting real transcripts — so the "background fraction" would
# then be a measure of biology, and any threshold on it would remove cells for
# expressing the wrong genes.
counts = adata.layers["counts"]
adata.obs["negctrl_counts"] = np.asarray(counts[:, is_background].sum(axis=1)).ravel()
adata.obs["other_nongene_counts"] = np.asarray(counts[:, is_other].sum(axis=1)).ravel()

# keep only real genes from here on; the summaries stay in obs as metrics
adata = adata[:, ~is_ctrl].copy()
sc.pp.calculate_qc_metrics(adata, percent_top=None, inplace=True, log1p=False)

empty = (adata.obs["total_counts"] == 0).to_numpy()
if empty.any():
    print(f"{empty.sum():,} cells ({100 * empty.mean():.2f}%) have zero gene counts "
          "after removing non-gene features.\n")

# THIS is the QC metric: what fraction of a cell's signal is measurable noise?
denom = adata.obs["total_counts"] + adata.obs["negctrl_counts"]
adata.obs["control_frac"] = np.where(
    denom > 0, adata.obs["negctrl_counts"] / denom.replace(0, np.nan), 0.0)

# This is NOT a QC metric — it is a biological signal, kept for the comparison
# below. Never filter on it.
denom2 = adata.obs["total_counts"] + adata.obs["other_nongene_counts"]
adata.obs["other_nongene_frac"] = np.where(
    denom2 > 0, adata.obs["other_nongene_counts"] / denom2.replace(0, np.nan), 0.0)

adata.obs["nucleus_ratio"] = adata.obs["nucleus_area"] / adata.obs["cell_area"].replace(0, np.nan)
adata.obs["counts_per_um2"] = adata.obs["total_counts"] / adata.obs["cell_area"].replace(0, np.nan)

print("median control_frac (true negative controls) : "
      f"{adata.obs['control_frac'].median():.6f}")
print("median other_nongene_frac (NOT quality)      : "
      f"{adata.obs['other_nongene_frac'].median():.4f}")
if adata.obs["other_nongene_frac"].median() > 20 * max(adata.obs["control_frac"].median(), 1e-9):
    print("\n-> The non-control features carry far more signal than the real")
    print("   controls. If you had filtered on them you would have removed cells")
    print("   for their biology. The comparison below shows where they sit.")

adata.obs[["total_counts", "n_genes_by_counts", "cell_area", "nucleus_ratio",
           "control_frac", "other_nongene_frac", "counts_per_um2"]].describe().round(4)

In [ ]:
metrics = [
    ("total_counts", "transcripts per cell", True),
    ("n_genes_by_counts", "genes detected per cell", False),
    ("cell_area", "cell area (µm²)", True),
    ("nucleus_ratio", "nucleus / cell area", False),
    ("control_frac", "negative-control fraction", False),
    ("other_nongene_frac", "other non-gene fraction (NOT quality)", False),
    ("counts_per_um2", "transcript density (per µm²)", False),
]

fig, axes = plt.subplots(2, 4, figsize=(17, 6))
for ax in axes.ravel()[len(metrics):]:
    ax.axis("off")
for ax, (col, label, logx) in zip(axes.ravel(), metrics):
    v = adata.obs[col].to_numpy()
    v = v[np.isfinite(v)]
    if logx:
        # Bin in log space. Plotting linear bins on a log axis gives bars of
        # wildly uneven width and squashes the left-hand tail into a wall.
        v = v[v > 0]
        bins = np.logspace(np.log10(v.min()), np.log10(v.max()), 60)
        ax.set_xscale("log")
    else:
        bins = 60
    ax.hist(v, bins=bins, color=NAVY)
    ax.set_xlabel(label); ax.set_ylabel("cells")
    ax.axvline(np.median(v), color=GOLD, lw=2)
sns.despine(); fig.suptitle("Per-cell QC (gold line = median)", y=1.02)
fig.tight_layout(); plt.show()

### Read the numbers out loud

**Median transcripts per cell.** Compare it with a 10x 3' scRNA-seq experiment
(5,000–20,000 UMIs). You have one to two orders of magnitude less. Consequences:

- per-cell expression of any single gene is close to binary — a cell has 0, 1 or 2
  copies of most transcripts
- gene–gene correlation within a cell is nearly meaningless
- but you have *many* cells and you know where they are, so **aggregate across space,
  not within cells**

That reframing — from "what does this cell express" to "what does this region
express" — is the single biggest mental shift for someone arriving from scRNA-seq.

**Nucleus/cell ratio near 1.0** means the segmentation gave that cell essentially no
cytoplasm. A pile-up at exactly 1.0 suggests nucleus-expansion segmentation rather
than true boundary detection.

## 3. Counts versus area — where segmentation failures show up

In a healthy dataset, bigger cells have more transcripts, roughly proportionally.
Deviations tell you specific stories.

In [ ]:
# Log axes cannot show zeros, and some cells genuinely have none. Drop them from
# THIS PLOT only, and report how many — a cell with zero gene counts is a QC
# finding, not just a plotting nuisance.
area = adata.obs["cell_area"].to_numpy()
counts = adata.obs["total_counts"].to_numpy()
ok = (area > 0) & (counts > 0)

n_zero_counts = int((counts <= 0).sum())
n_zero_area = int((area <= 0).sum())
if n_zero_counts:
    print(f"{n_zero_counts:,} cells ({100 * n_zero_counts / adata.n_obs:.2f}%) have zero "
          f"gene counts and cannot be shown on a log axis.")
    print("These are usually polygons whose only counts were on control features,")
    print("or empty segmentations. They will be removed by the filter in section 5.")
if n_zero_area:
    print(f"{n_zero_area:,} cells have zero area — check the segmentation output.")

# hexbin must be told to bin in log space itself. Calling ax.set_xscale("log")
# afterwards bins linearly and then stretches the hexagons, which smears
# everything into one dark rectangle.
fig, ax = plt.subplots(figsize=(6.5, 5.5))
h = ax.hexbin(area[ok], counts[ok],
              xscale="log", yscale="log",          # <- the important part
              gridsize=70, bins="log", cmap="viridis", mincnt=1)
ax.set_xlabel("cell area (µm²)"); ax.set_ylabel("transcripts per cell")
plt.colorbar(h, ax=ax, label="log10(cells per bin)")
ax.set_title(f"counts vs area  ({int(ok.sum()):,} cells shown)")

# median counts per area decile, to show the trend through the cloud
sub = adata.obs.loc[ok]
q = pd.qcut(sub["cell_area"], 12, labels=False, duplicates="drop")
trend = sub.groupby(q, observed=True).agg(
    area=("cell_area", "median"), counts=("total_counts", "median"))
ax.plot(trend["area"], trend["counts"], color=CORAL, lw=2.2,
        marker="o", ms=4, label="median per area decile")
ax.legend(loc="lower right", framealpha=0.9)
plt.show()

r = np.corrcoef(np.log10(area[ok]), np.log10(counts[ok]))[0, 1]
print(f"\ncorrelation of log(area) with log(counts): r = {r:.2f}")

Four regions of this plot, and what each means:

| Where | Likely cause |
|---|---|
| bottom-left (small + empty) | debris, cut nuclei at the section edge, failed segmentation |
| not shown at all | cells with **zero** gene counts — a log axis cannot display them, so the cell above counts them for you |
| top-right (large + rich) | **merged cells** — two cells in one polygon (the spatial doublet) |
| bottom-right (large + empty) | polygon drawn over extracellular space, or a fat adipocyte-like cell |
| main diagonal | ordinary cells |

Note that "spatial doublet" is *not* the same as a scRNA-seq doublet, and the tools
are different: `scrublet`/`DoubletFinder` model two transcriptomes summed in a
droplet. Here you have a geometrical failure, and the fix is geometric — look at the
polygon, or re-segment. **Do not run scRNA-seq doublet detection on Xenium data and
believe the output.**

The orange line is the median counts per area decile. In healthy data it climbs
steadily; a plateau at the right-hand end means the largest polygons are *not*
gaining transcripts in proportion, which is what you would expect if they are
enclosing empty space rather than more cell.


## 4. The step people skip: put QC on the tissue

If your low-count cells sit in one stripe, that is an instrument or focus problem.
If they sit in a coherent tissue region, they may be real biology — necrosis,
stroma, a dense lymphoid aggregate — and filtering them silently deletes a
structure from your paper.

In [ ]:
def spatial_plot(adata, color, ax=None, s=1.2, cmap="viridis", vmin=None, vmax=None,
                 title=None, categorical=False, legend=False):
    """Minimal spatial scatter. Deliberately hand-rolled so you can see the mechanics:
    it is just obsm['spatial'] with a colour."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5.5, 5.5))
    x, y = adata.obsm["spatial"].T
    if categorical:
        cats = adata.obs[color].astype("category")
        codes = cats.cat.codes
        cm = plt.get_cmap("tab20")
        ax.scatter(x, y, s=s, c=[cm(i % 20) for i in codes], linewidths=0, rasterized=True)
        if legend:
            for i, name in enumerate(cats.cat.categories):
                ax.scatter([], [], s=25, color=cm(i % 20), label=name)
            ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=8)
    else:
        v = adata.obs[color].to_numpy() if color in adata.obs else np.asarray(
            adata[:, color].X.todense()).ravel()
        pc = ax.scatter(x, y, s=s, c=v, cmap=cmap, vmin=vmin, vmax=vmax,
                        linewidths=0, rasterized=True)
        plt.colorbar(pc, ax=ax, fraction=0.046, label=color)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title or color)
    return ax

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))
spatial_plot(adata, "total_counts", ax=axes[0], vmax=np.percentile(adata.obs["total_counts"], 98),
             title="transcripts per cell")
spatial_plot(adata, "cell_area", ax=axes[1], vmax=np.percentile(adata.obs["cell_area"], 98),
             title="cell area")
spatial_plot(adata, "control_frac", ax=axes[2], vmax=np.percentile(adata.obs["control_frac"], 98),
             cmap="magma", title="control fraction")
plt.tight_layout(); plt.show()

> **Try it yourself — colour the tissue by anything**
>
> The three maps above use three QC columns. Every column in `adata.obs` can be
> plotted this way. Pick a different one and see whether it has spatial structure.

In [ ]:
# What can you colour by? These are the numeric columns available.
numeric_cols = [c for c in adata.obs.columns
                if pd.api.types.is_numeric_dtype(adata.obs[c])]
print(numeric_cols)

In [ ]:
COLOUR_BY = "n_genes_by_counts"      # <-- CHANGE THIS to any name printed above

spatial_plot(adata, COLOUR_BY,
             vmax=np.percentile(adata.obs[COLOUR_BY].dropna(), 98),
             title=COLOUR_BY)
plt.show()

print(adata.obs[COLOUR_BY].describe().round(2))

### What to look for, concretely

- **Straight edges, stripes or a grid.** Xenium images in fields of view (FOVs) and
  stitches them. A visible tiling pattern is technical, full stop.
- **A gradient across the section.** Often focus, tissue thickness, or permeabilisation.
- **A blob of low counts with a soft boundary.** Usually biological — necrotic core,
  dense collagen, adipose.
- **Anything that follows a fold or a tear.** Section damage; exclude the region
  explicitly rather than letting a count threshold do it invisibly.

### Exercise 2.2
Is there a region of this crop you would exclude entirely, before any per-cell
filtering? Draw its bounding box, count how many cells it holds, and justify it in
one sentence you would be willing to put in a methods section.

In [ ]:
# your code here — a rectangle mask on adata.obsm["spatial"] is enough

## 5. Choosing filters

Common starting points for Xenium (not laws — starting points):

| Filter | Typical | Why |
|---|---|---|
| `total_counts >= 10` | 10–25 | below this, a cell has no usable profile |
| `n_genes_by_counts >= 5` | 5–10 | catches empty polygons |
| `cell_area` within 1st–99th pct | | removes debris and giant merges |
| `control_frac <= 0.05` | | drops cells that are mostly background |

Set them, then **look at what you removed in space** before accepting them.

In [ ]:
MIN_COUNTS = 10
MIN_GENES = 5
MAX_CONTROL_FRAC = 0.05     # on TRUE negative controls this is very lax;
                            # a clean run sits around 1e-4
area_lo, area_hi = np.percentile(adata.obs["cell_area"], [1, 99])

keep = (
    (adata.obs["total_counts"] >= MIN_COUNTS)
    & (adata.obs["n_genes_by_counts"] >= MIN_GENES)
    & (adata.obs["control_frac"] <= MAX_CONTROL_FRAC)
    & (adata.obs["cell_area"].between(area_lo, area_hi))
).to_numpy()

adata.obs["qc_pass"] = keep
print(f"keeping {keep.sum():,} of {adata.n_obs:,} cells ({100*keep.mean():.1f}%)")
for name, m in [
    ("low counts", adata.obs["total_counts"] < MIN_COUNTS),
    ("few genes", adata.obs["n_genes_by_counts"] < MIN_GENES),
    ("high control frac", adata.obs["control_frac"] > MAX_CONTROL_FRAC),
    ("area outlier", ~adata.obs["cell_area"].between(area_lo, area_hi)),
]:
    print(f"  removed by {name:<20} {int(np.asarray(m).sum()):>7,}")

> **Try it yourself — how strict is too strict?**
>
> Change `TRY_MIN_COUNTS` and re-run. You are not changing the real filter above —
> this cell only reports what *would* happen, so experiment freely.
>
> Watch two numbers: how many cells you keep, and how the *smallest* cells fare.
> Immune cells are small. A threshold that looks harmless can delete them.

In [ ]:
TRY_MIN_COUNTS = 10        # <-- CHANGE THIS (try 5, 25, 50)

would_keep = (adata.obs["total_counts"] >= TRY_MIN_COUNTS).to_numpy()
pct = 100 * would_keep.mean()
print(f"threshold {TRY_MIN_COUNTS:>3}: keep {would_keep.sum():>7,} cells ({pct:.1f}%)")

# who gets removed? compare the size of kept vs removed cells
kept_area = adata.obs.loc[would_keep, "cell_area"].median()
lost_area = adata.obs.loc[~would_keep, "cell_area"].median()
print(f"median cell area — kept {kept_area:6.1f} um2, removed {lost_area:6.1f} um2")
if lost_area < kept_area:
    print("The cells you remove are SMALLER than average. Which cell types are small?")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
x, y = adata.obsm["spatial"].T
axes[0].scatter(x[keep], y[keep], s=1, c=ICE, linewidths=0, rasterized=True)
axes[0].scatter(x[~keep], y[~keep], s=1.6, c=CORAL, linewidths=0, rasterized=True)
axes[0].set_title("kept (blue) vs discarded (orange)")

axes[1].hexbin(x[~keep], y[~keep], gridsize=45, cmap="Oranges", mincnt=1)
axes[1].set_title("density of discarded cells")
for ax in axes:
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

**The question that matters:** do the orange cells form a structure?

If discarded cells cluster into a coherent anatomical region, your filter is not a
quality filter — it is a biology filter, and you have just deleted a cell population.
The honest options are (a) relax the threshold and handle the region separately, or
(b) keep the threshold and say explicitly in the methods which region it removed.

This check has no equivalent in scRNA-seq, because there is nowhere to plot it.

### Which filter is doing it?

"The discarded cells form a structure" is the finding. **Which filter caught them** is
the actionable part, because the four filters fail for entirely different reasons.

The next cell maps each one separately.

In [ ]:
reasons = {
    "low counts": (adata.obs["total_counts"] < MIN_COUNTS).to_numpy(),
    "few genes": (adata.obs["n_genes_by_counts"] < MIN_GENES).to_numpy(),
    "high control frac": (adata.obs["control_frac"] > MAX_CONTROL_FRAC).to_numpy(),
    "area too small": (adata.obs["cell_area"] < area_lo).to_numpy(),
    "area too large": (adata.obs["cell_area"] > area_hi).to_numpy(),
}

fig, axes = plt.subplots(1, len(reasons), figsize=(4.0 * len(reasons), 4.3))
for ax, (name, m) in zip(np.atleast_1d(axes), reasons.items()):
    ax.scatter(x, y, s=0.5, c="0.9", linewidths=0, rasterized=True)
    ax.scatter(x[m], y[m], s=1.6, c=CORAL, linewidths=0, rasterized=True)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{name}\n{m.sum():,} cells ({100 * m.mean():.1f}%)", fontsize=10)
plt.tight_layout(); plt.show()

overlap = pd.DataFrame({
    "removed by this": {k: int(v.sum()) for k, v in reasons.items()},
    "ONLY this one": {k: int((v & ~np.logical_or.reduce(
        [w for j, w in reasons.items() if j != k])).sum()) for k, v in reasons.items()},
})
display(overlap)

### Reading the panels

Whichever panel reproduces the structure you saw is the one to think about.

**Low counts / few genes over a tumour nest.** The usual cause is not bad data but
**dense packing**. Tumour cells in a nest are tightly apposed with little cytoplasm,
so segmentation gives small polygons that capture few transcripts. The measurement is
working; the cells are genuinely small compartments.

A second contributor in very dense, highly expressing regions is **optical crowding**:
transcripts sitting so close that the decoder cannot resolve them, so they fail the
QV filter and never reach the matrix. That shows up as a *relative* drop in counts
exactly where expression is highest, which is counter-intuitive and real.

**Area too small over a nest.** Same mechanism, seen through the other filter.

**Area too large.** Merged polygons — two or more cells in one boundary. Expect this
at the *edges* of dense regions rather than in their cores.

**High control fraction over a region.** Different in kind. This one is genuinely
about signal quality: those cells have proportionally more background. If it maps to a
tissue fold or a section edge, exclude the region explicitly rather than by threshold.

**Necrosis** is the other real possibility for a tumour core: degraded RNA, genuinely
low counts, and a biologically meaningful region you probably want to keep and label
rather than silently delete.

In [ ]:
# Compare REGIONS, not kept-vs-discarded. Comparing discarded cells against kept
# ones on a count-derived metric is circular: the filter selected them on counts,
# so of course they look sparse. Instead, find the tiles where discarding is
# concentrated and compare EVERY cell there against every cell elsewhere.
BIN = 75.0        # um; big enough that each tile holds a few dozen cells

gx = ((x - x.min()) // BIN).astype(int)
gy = ((y - y.min()) // BIN).astype(int)
tile = pd.Series(list(zip(gx, gy)), index=adata.obs_names)

disc = ~keep
rate = (pd.DataFrame({"tile": tile.to_numpy(), "disc": disc})
        .groupby("tile")["disc"].agg(["mean", "size"]))
rate = rate[rate["size"] >= 20]
hot = set(rate[rate["mean"] >= rate["mean"].quantile(0.90)].index)
in_hot = tile.isin(hot).to_numpy()

print(f"high-discard tiles hold {in_hot.sum():,} cells; discard rate there "
      f"{100 * disc[in_hot].mean():.1f}% vs {100 * disc[~in_hot].mean():.1f}% elsewhere\n")

cols = ["cell_area", "counts_per_um2", "total_counts", "nucleus_ratio",
        "control_frac", "other_nongene_frac"]
cmp = pd.DataFrame({
    "high-discard region": adata.obs.loc[in_hot, cols].median(),
    "rest of section": adata.obs.loc[~in_hot, cols].median(),
}).round(3)
cmp["ratio"] = (cmp["high-discard region"] / cmp["rest of section"]).round(2)
display(cmp)

area_r = cmp.loc["cell_area", "ratio"]
dens_r = cmp.loc["counts_per_um2", "ratio"]
cf_r = cmp.loc["control_frac", "ratio"]
other_r = cmp.loc["other_nongene_frac", "ratio"]
if other_r > 2 and cf_r < 2:
    print("NOTE: the NON-control non-gene features are strongly enriched here")
    print(f"      (ratio {other_r:.1f}) while true background is not. That is a")
    print("      biological signal — see the section below before filtering.\n")

print("Interpretation:")
if area_r < 0.8 and dens_r > 0.8:
    print("  Cells there are SMALLER but just as transcript-dense. The tissue is")
    print("  fine; your filter is removing cells for being small. Packing and")
    print("  segmentation, not data quality.")
elif dens_r < 0.8 and cf_r < 1.5:
    print("  The tissue there yields less per unit area. Think necrosis, poor")
    print("  fixation, or a fold — a real regional difference in the sample.")
elif cf_r > 1.5:
    print("  Negative-control background is proportionally higher there. This one IS")
    print("  a signal-quality problem; consider excluding the region explicitly.")
else:
    print("  No clear geometric explanation — look at the region in Xenium Explorer.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
axes[0].scatter(x, y, s=0.5, c="0.88", linewidths=0, rasterized=True)
axes[0].scatter(x[in_hot], y[in_hot], s=1.4, c=NAVY, linewidths=0, rasterized=True)
axes[0].set_title("the region being compared")

pc = axes[1].scatter(x, y, c=adata.obs["cell_area"], s=1.0, cmap="viridis",
                     vmax=np.percentile(adata.obs["cell_area"].dropna(), 98),
                     linewidths=0, rasterized=True)
plt.colorbar(pc, ax=axes[1], label="cell area (µm²)")
axes[1].set_title("cell area — does it match?")
for ax in axes:
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

### If `control_frac` is what flagged the region — read this

A high **negative-control** fraction is a genuine quality problem: those cells really
do have proportionally more measurable noise.

But if you computed that fraction over *every* non-gene feature, you did not measure
noise. Deprecated codewords are retired probes that still detect **real transcripts**,
and they carry orders of magnitude more counts than the true controls. A region rich
in them is a region expressing the genes those probes were designed for — biology, not
background.

The arithmetic on this section: the true negative controls carry a few dozen counts in
total, giving a section-wide background fraction around 1e-5. The deprecated codewords
carry hundreds of thousands, giving a fraction near 0.1 — four orders of magnitude
larger, and concentrated wherever those retired genes are expressed.

So a threshold of 0.05 on the wrong quantity removes every cell whose retired-probe
signal exceeds 5% of its counts. In a tumour section that is the tumour.

**The cells above now compute both separately:** `control_frac` from the two
negative-control classes (filter on this), and `other_nongene_frac` from everything
else (never filter on this — but do look at where it is high, because it is telling
you something real about the tissue).

In [ ]:
# Where does the non-control signal sit? If it maps onto a tissue structure,
# that is biology, and it is worth knowing which retired genes are involved.
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
for ax, col, ttl in [(axes[0], "control_frac", "negative-control fraction\n(a quality metric)"),
                     (axes[1], "other_nongene_frac", "other non-gene fraction\n(NOT a quality metric)")]:
    v = adata.obs[col].to_numpy()
    pc = ax.scatter(x, y, c=v, s=1.1, cmap="magma",
                    vmax=max(np.percentile(v[np.isfinite(v)], 99), 1e-6),
                    linewidths=0, rasterized=True)
    plt.colorbar(pc, ax=ax, fraction=0.046)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(ttl, fontsize=10)
plt.tight_layout(); plt.show()

print(f"negative-control fraction   median {adata.obs['control_frac'].median():.6f}, "
      f"99th pct {adata.obs['control_frac'].quantile(0.99):.5f}")
print(f"other non-gene fraction     median {adata.obs['other_nongene_frac'].median():.4f}, "
      f"99th pct {adata.obs['other_nongene_frac'].quantile(0.99):.3f}")

### So what do you actually do?

Four defensible options. They are not equally good, and the choice depends on your
question.

**1. Keep the threshold, name the casualty.** Perfectly acceptable *if you say so*:
"cells with <10 transcripts were excluded; these were enriched in densely packed
tumour nests, so tumour cell frequencies are underestimated." A reader can then
discount your composition estimates appropriately.

**2. Lower the threshold and accept noisier cells.** Reasonable if you care about
*where* tumour cells are rather than what each one expresses. Position survives low
counts far better than expression does.

**3. Threshold on density, not on counts.** `counts_per_um2` asks whether a cell
yielded a reasonable amount *for its size*, which does not penalise small cells.
Try it as an alternative filter and see whether the structure disappears.

**4. Filter within region.** Compute thresholds separately inside and outside dense
regions. Most defensible, most work, and hardest to explain in a methods section.

**What is not acceptable** is option 1 without the second half of the sentence.

> **The general lesson.** Every QC threshold is a hypothesis that low-quality cells
> are randomly distributed. Spatial data lets you test that hypothesis, and here it
> failed. In scRNA-seq the same thing happens — dense, small, fragile cells are lost
> at dissociation and at filtering — and you simply cannot see it.

### Exercise 2.3b
Re-filter using `counts_per_um2` instead of `total_counts`, at a threshold that
removes a similar total number of cells. Does the spatial structure in the discarded
set weaken? If it does, your original filter was measuring cell size more than data
quality.

In [ ]:
# your code here

In [ ]:
adata = adata[keep].copy()
adata.write_h5ad(DATA / "ovarian_qc.h5ad", compression="gzip")
print(adata)

### Exercise 2.3
Redo the filtering with `MIN_COUNTS = 25`. How many cells do you lose, and — the
real question — do you lose them evenly across the tissue, or does one region take
the hit? Keep the plot; you will want it in notebook 06.

---
**Next:** `03_from_counts_to_cell_types.ipynb`